##Data Lead
Import and Inspect:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA = '/content/drive/MyDrive/NYPA PowerHack/data'
print(os.listdir(DATA))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['README.md', 'emissions_factors.csv', 'nyiso_fuel_mix_daily_2023_2025.csv', 'nyiso_fuel_mix_monthly_2023_2025.csv', 'nyiso_hourly_profile_by_month_2023_2025.csv', 'clean']


In [ ]:
import pandas as pd

monthly = pd.read_csv(f'{DATA}/nyiso_fuel_mix_monthly_2023_2025.csv')
daily   = pd.read_csv(f'{DATA}/nyiso_fuel_mix_daily_2023_2025.csv', parse_dates=['date'])
hourly  = pd.read_csv(f'{DATA}/nyiso_hourly_profile_by_month_2023_2025.csv')
ef      = pd.read_csv(f'{DATA}/emissions_factors.csv')

for name, df in [('monthly', monthly), ('daily', daily), ('hourly', hourly), ('emissions', ef)]:
    print(name, df.shape, '| missing values:', df.isna().sum().sum())


OUT = f'{DATA}/clean'
os.makedirs(OUT, exist_ok=True)

fuels = ['natural_gas','dual_fuel','nuclear','hydro','wind','other_renewables','other_fossil']
fossil_fuels = ['natural_gas','dual_fuel','other_fossil']
zero_fuels   = ['nuclear','hydro','wind','other_renewables']
mwh_cols = [f + '_mwh' for f in fuels]

def add_groups_and_shares(df, suffix, total_col):
    df = df.copy()
    df['fossil' + suffix]      = sum(df[f + suffix] for f in fossil_fuels)
    df['zero_carbon' + suffix] = sum(df[f + suffix] for f in zero_fuels)
    for f in fuels:
        df[f + '_share_pct'] = (df[f + suffix] / df[total_col] * 100).round(2)
    df['fossil_share_pct']      = (df['fossil' + suffix] / df[total_col] * 100).round(2)
    df['zero_carbon_share_pct'] = (df['zero_carbon' + suffix] / df[total_col] * 100).round(2)
    return df

# monthly
monthly_clean = monthly.copy()
monthly_clean.insert(0, 'date', pd.to_datetime(dict(year=monthly.year, month=monthly.month, day=1)).dt.strftime('%Y-%m-%d'))
monthly_clean['days_in_month'] = pd.to_datetime(monthly_clean['date']).dt.days_in_month
monthly_clean['total_twh'] = (monthly_clean['total_mwh'] / 1e6).round(3)
monthly_clean = add_groups_and_shares(monthly_clean, '_mwh', 'total_mwh')

# daily and hourly
daily_clean  = add_groups_and_shares(daily, '_mwh', 'total_mwh')
hourly_clean = add_groups_and_shares(hourly, '_avg_mw', 'total_avg_mw')

# yearly: sum MWh first, then divide
y = monthly_clean.groupby('year')[mwh_cols + ['total_mwh', 'fossil_mwh', 'zero_carbon_mwh']].sum().reset_index()
yearly = y[['year'] + mwh_cols + ['total_mwh']].copy()
for f in fuels:
    yearly[f + '_share_pct'] = (y[f + '_mwh'] / y['total_mwh'] * 100).round(2)
yearly['fossil_share_pct']      = (y['fossil_mwh'] / y['total_mwh'] * 100).round(2)
yearly['zero_carbon_share_pct'] = (y['zero_carbon_mwh'] / y['total_mwh'] * 100).round(2)

# checks
share_cols = [f + '_share_pct' for f in fuels]
for name, df in [('monthly', monthly_clean), ('daily', daily_clean), ('hourly', hourly_clean), ('yearly', yearly)]:
    assert df[share_cols].sum(axis=1).between(99.9, 100.1).all(), f'{name}: shares do not add to 100%'
    assert (df['fossil_share_pct'] + df['zero_carbon_share_pct']).between(99.9, 100.1).all(), f'{name}: fossil + zero-carbon != 100%'
print('All checks passed: shares add to 100% in every row.')

# save to Drive
monthly_clean.to_csv(f'{OUT}/monthly_clean.csv', index=False)
daily_clean.to_csv(f'{OUT}/daily_clean.csv', index=False)
hourly_clean.to_csv(f'{OUT}/hourly_clean.csv', index=False)
yearly.to_csv(f'{OUT}/yearly_shares.csv', index=False)
print('Saved to:', OUT)
print(os.listdir(OUT))

yearly[['year'] + share_cols + ['fossil_share_pct']]

monthly (36, 12) | missing values: 0
daily (1096, 15) | missing values: 0
hourly (864, 13) | missing values: 0
emissions (7, 8) | missing values: 0
All checks passed: shares add to 100% in every row.
Saved to: /content/drive/MyDrive/NYPA PowerHack/data/clean
['monthly_clean.csv', 'yearly_shares.csv', 'hourly_clean.csv', 'daily_clean.csv', 'emissions_factors_clean.csv']


,year,natural_gas_share_pct,dual_fuel_share_pct,nuclear_share_pct,hydro_share_pct,wind_share_pct,other_renewables_share_pct,other_fossil_share_pct,fossil_share_pct
0,2023,23.28,27.26,22.06,21.75,3.75,1.78,0.14,50.67
1,2024,24.70,27.01,20.76,20.64,4.68,2.10,0.11,51.82
2,2025,25.09,27.54,21.39,18.21,5.35,2.27,0.15,52.78


In [ ]:
display(monthly.head())
print(monthly.info())
print(monthly.shape)
print()
display(daily.head())
print(daily.info())
print(daily.shape)
print()
display(hourly.head())
print(hourly.info())
print(hourly.shape)
print()
display(ef.head())
print(ef.info())
print(ef.shape)

,year,month,month_name,season,natural_gas_mwh,dual_fuel_mwh,nuclear_mwh,hydro_mwh,wind_mwh,other_renewables_mwh,other_fossil_mwh,total_mwh
0,2023,1,Jan,Winter,2647071,2230614,2471425,2515014,317117,178484,6363,10366088
1,2023,2,Feb,Winter,1965276,2518621,2187373,2166940,520180,136344,20012,9514746
2,2023,3,Mar,Spring,1829051,2362288,2143697,2349213,489949,162649,11565,9348412
3,2023,4,Apr,Spring,1794058,2063199,1772552,2231050,447304,147844,3504,8459511
4,2023,5,May,Spring,1914483,2250960,2458572,2300320,330457,204641,2064,9461497


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   year                  36 non-null     int64 
 1   month                 36 non-null     int64 
 2   month_name            36 non-null     object
 3   season                36 non-null     object
 4   natural_gas_mwh       36 non-null     int64 
 5   dual_fuel_mwh         36 non-null     int64 
 6   nuclear_mwh           36 non-null     int64 
 7   hydro_mwh             36 non-null     int64 
 8   wind_mwh              36 non-null     int64 
 9   other_renewables_mwh  36 non-null     int64 
 10  other_fossil_mwh      36 non-null     int64 
 11  total_mwh             36 non-null     int64 
dtypes: int64(10), object(2)
memory usage: 3.5+ KB
None
(36, 12)



,date,year,month,month_name,day_of_week,is_weekend,season,natural_gas_mwh,dual_fuel_mwh,nuclear_mwh,hydro_mwh,wind_mwh,other_renewables_mwh,other_fossil_mwh,total_mwh
0,2023-01-01,2023,1,Jan,Sunday,1,Winter,70038,64101,80281,78391,17374,5880,120,316185
1,2023-01-02,2023,1,Jan,Monday,0,Winter,76081,69851,80288,80852,7568,5298,153,320091
2,2023-01-03,2023,1,Jan,Tuesday,0,Winter,83474,79857,80280,77567,4878,5313,187,331556
3,2023-01-04,2023,1,Jan,Wednesday,0,Winter,79063,73132,80269,76365,9849,5666,183,324527
4,2023-01-05,2023,1,Jan,Thursday,0,Winter,88853,75305,80260,80216,9641,5915,168,340358


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  1096 non-null   datetime64[ns]
 1   year                  1096 non-null   int64         
 2   month                 1096 non-null   int64         
 3   month_name            1096 non-null   object        
 4   day_of_week           1096 non-null   object        
 5   is_weekend            1096 non-null   int64         
 6   season                1096 non-null   object        
 7   natural_gas_mwh       1096 non-null   int64         
 8   dual_fuel_mwh         1096 non-null   int64         
 9   nuclear_mwh           1096 non-null   int64         
 10  hydro_mwh             1096 non-null   int64         
 11  wind_mwh              1096 non-null   int64         
 12  other_renewables_mwh  1096 non-null   int64         
 13  other_fossil_mwh  

,year,month,month_name,hour_beginning,hour_label,natural_gas_avg_mw,dual_fuel_avg_mw,nuclear_avg_mw,hydro_avg_mw,wind_avg_mw,other_renewables_avg_mw,other_fossil_avg_mw,total_avg_mw
0,2023,1,Jan,0,00:00,2749,2426,3326,2849,434,235,5,12024
1,2023,1,Jan,1,01:00,2598,2302,3327,2794,449,235,5,11710
2,2023,1,Jan,2,02:00,2544,2267,3327,2775,414,235,5,11567
3,2023,1,Jan,3,03:00,2576,2272,3327,2752,435,234,5,11601
4,2023,1,Jan,4,04:00,2674,2321,3327,2755,454,234,5,11770


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864 entries, 0 to 863
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   year                     864 non-null    int64 
 1   month                    864 non-null    int64 
 2   month_name               864 non-null    object
 3   hour_beginning           864 non-null    int64 
 4   hour_label               864 non-null    object
 5   natural_gas_avg_mw       864 non-null    int64 
 6   dual_fuel_avg_mw         864 non-null    int64 
 7   nuclear_avg_mw           864 non-null    int64 
 8   hydro_avg_mw             864 non-null    int64 
 9   wind_avg_mw              864 non-null    int64 
 10  other_renewables_avg_mw  864 non-null    int64 
 11  other_fossil_avg_mw      864 non-null    int64 
 12  total_avg_mw             864 non-null    int64 
dtypes: int64(11), object(2)
memory usage: 87.9+ KB
None
(864, 13)



,fuel_column,fuel_label,co2_lb_per_mmbtu,heat_rate_btu_per_kwh,central_tco2_per_mwh,low_tco2_per_mwh,high_tco2_per_mwh,basis
0,natural_gas,Natural Gas,116.65,7535,0.399,0.399,0.581,EIA natural gas CO2 coefficient x 2025 avg hea...
1,dual_fuel,Dual Fuel,116.65,10293,0.545,0.399,0.765,EIA natural gas CO2 coefficient x 2025 avg hea...
2,nuclear,Nuclear,0.00,0,0.000,0.000,0.000,No direct combustion emissions
3,hydro,Hydro,0.00,0,0.000,0.000,0.000,No direct combustion emissions
4,wind,Wind,0.00,0,0.000,0.000,0.000,No direct combustion emissions


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   fuel_column            7 non-null      object 
 1   fuel_label             7 non-null      object 
 2   co2_lb_per_mmbtu       7 non-null      float64
 3   heat_rate_btu_per_kwh  7 non-null      int64  
 4   central_tco2_per_mwh   7 non-null      float64
 5   low_tco2_per_mwh       7 non-null      float64
 6   high_tco2_per_mwh      7 non-null      float64
 7   basis                  7 non-null      object 
dtypes: float64(4), int64(1), object(3)
memory usage: 580.0+ bytes
None
(7, 8)


In [40]:
print(monthly.describe())
print()

print(monthly_clean['natural_gas_mwh'].min())
print(monthly_clean['natural_gas_mwh'].max())
print()

print(monthly_clean['dual_fuel_mwh'].min())
print(monthly_clean['dual_fuel_mwh'].max())
print()

print(monthly_clean['nuclear_mwh'].min())
print(monthly_clean['nuclear_mwh'].max())
print()

print(monthly_clean['hydro_mwh'].min())
print(monthly_clean['hydro_mwh'].max())
print()

print(monthly_clean['wind_mwh'].min())
print(monthly_clean['wind_mwh'].max())
print()

print(monthly_clean['other_renewables_mwh'].min())
print(monthly_clean['other_renewables_mwh'].max())
print()

print(monthly_clean['other_fossil_mwh'].min())
print(monthly_clean['other_fossil_mwh'].max())

              year     month  natural_gas_mwh  dual_fuel_mwh   nuclear_mwh  \
count    36.000000  36.00000     3.600000e+01   3.600000e+01  3.600000e+01   
mean   2024.000000   6.50000     2.633036e+06   2.945922e+06  2.310859e+06   
std       0.828079   3.50102     4.938909e+05   8.109995e+05  1.964881e+05   
min    2023.000000   1.00000     1.794058e+06   1.973903e+06  1.716707e+06   
25%    2023.000000   3.75000     2.253082e+06   2.282950e+06  2.228512e+06   
50%    2024.000000   6.50000     2.646520e+06   2.700882e+06  2.382816e+06   
75%    2025.000000   9.25000     2.892037e+06   3.399086e+06  2.436628e+06   
max    2025.000000  12.00000     3.773275e+06   5.272367e+06  2.488093e+06   

          hydro_mwh       wind_mwh  other_renewables_mwh  other_fossil_mwh  \
count  3.600000e+01      36.000000             36.000000          36.00000   
mean   2.178040e+06  497952.000000         222038.750000       14427.00000   
std    2.217917e+05  191414.904436          41069.450102       